In [ ]:
#load all the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score, roc_auc_score


In [ ]:
# Separate features and target
X_train=pd.read_csv('../output/x_train.csv')
X_test=pd.read_csv('../output/x_test.csv')
y_train=pd.read_csv('../output/y_train.csv').squeeze()
y_test=pd.read_csv('../output/y_test.csv')

# Apply Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Test different values of K
k_values = [3, 5, 7, 9, 11, 15]
scores_of_diferent_k_values = []

# Knn forloop to test different k values
for current_k_value in k_values:
    knn_model = KNeighborsClassifier(n_neighbors=current_k_value)
    knn_model.fit(X_train_scaled, y_train)
    y_predicted = knn_model.predict(X_test_scaled)
    
    # Measures the overall percentage of wine samples the model correctly assigned to the right quality rating.
    accuracy_of_the_current_k_value = accuracy_score(y_test, y_predicted)
    f1_score_of_the_current_k_value = f1_score(y_test, y_predicted, average='weighted') 
    #Calculates how many wines the model labeled as a specific quality, belonged to that grade.
    precision_of_the_current_k_value = precision_score(y_test, y_predicted, average='weighted', zero_division=0)
    # true poritive
    recall_of_the_current_k_value = recall_score(y_test, y_predicted, average='weighted')

    scores_of_diferent_k_values.append((current_k_value, accuracy_of_the_current_k_value, f1_score_of_the_current_k_value, precision_of_the_current_k_value, recall_of_the_current_k_value))
    print(f"K={current_k_value} -> Accuracy: {accuracy_of_the_current_k_value:.4f}, F1: {f1_score_of_the_current_k_value:.4f}, Precision: {precision_of_the_current_k_value:.4f} Recall: {recall_of_the_current_k_value:.4f}")

In [ ]:
# Convert scores to Data Frame for easy plotting
knn_results_data_frame = pd.DataFrame(scores_of_diferent_k_values, columns=['K', 'Accuracy', 'F1-Score', 'Precision', 'Recall'])

# Plot the results
plt.figure(figsize=(10, 5))
plt.plot(knn_results_data_frame['K'], knn_results_data_frame['Accuracy'], marker='o', label='Accuracy')
plt.plot(knn_results_data_frame['K'], knn_results_data_frame['F1-Score'], marker='s', label='F1-Score')
plt.title('KNN Performance vs K-Value')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Score')
plt.legend()
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance

# Use the best K value(11)
knn_final_model_wiht_the_most_accurate_k_value = KNeighborsClassifier(n_neighbors=11)
knn_final_model_wiht_the_most_accurate_k_value.fit(X_train_scaled, y_train)

# This identifies which physicochemical properties influence the quality prediction most
result = permutation_importance(knn_final_model_wiht_the_most_accurate_k_value, X_test_scaled, y_test, n_repeats=10, random_state=42)

# Organize the data
feature_importance_in_ascending_order = pd.Series(result.importances_mean, index=X_train.columns).sort_values(ascending=True)

# Plot the results
plt.figure(figsize=(10, 6))
feature_importance_in_ascending_order.plot(kind='barh', color='salmon')
plt.xlabel("Mean Importance")
plt.title("Feature Importance in KNN")
plt.show()